# Profile similarities

## Replication (paper-like)
Vector: `p_paper_vector` vs `c_paper_vector`  
Metrics:
- `paper_wc` (weighted cosine on Schwartz circumplex)
- `paper_cos` (cosine)
- `paper_tau` (Kendall τ on ranks, tie-aware)

## Extension (probabilistic)
Vector: `p_prob_vector` vs `c_prob_vector`  
Metrics:
- `prob_cos` (cosine on intensities)
- `prob_wc` (weighted cosine on intensities, circumplex-aware)
- `prob_jsd` (Jensen–Shannon divergence on **normalized** prob profiles)

Outputs:
- `parent_child_profile_similarity_clean.csv`
- `parent_child_profile_similarity_summary_clean.csv`


In [ ]:
#!pip install pingouin
import matplotlib.pyplot as plt
import ast, json, math, os, glob, csv, sys, re
import numpy as np
import pandas as pd
import pingouin as pg
from scipy.stats import ttest_ind, kendalltau, rankdata

sys.path.append("/Proyecto/Value-disagreement/Python/Utilities")
import Dict_Object

In [ ]:
DATA_PATH = "/Proyecto/Value-disagreement/Python/Models/Inference/final_profiles/deba_with_profiles.csv"
SEP = "|"
NUM_VALUES = 10

# Vector order (Paper Circumplex order)
VALUES = Dict_Object.ValueConstants.SCHWARTZ_VALUES
VALUES_ORDER = [VALUES[i] for i in Dict_Object.ValueConstants.SCHWARTZ_VALUES_CIRCUMPLEX_ORDER]

# test new order (try)
CIRC_ORDER = [VALUES_ORDER.index(v) for v in VALUES_ORDER]

# Weight Matrices (Paper Circumplex order)
circumplex_weight_matrix_5 = Dict_Object.ValueConstants.SCHWARTZ_VALUE_SIMILARITY_MATRIX_5   #sigma = 5
circumplex_weight_matrix_2 = Dict_Object.ValueConstants.SCHWARTZ_VALUE_SIMILARITY_MATRIX_2   #sigma = 2
circumplex_weight_matrix_1 = Dict_Object.ValueConstants.SCHWARTZ_VALUE_SIMILARITY_MATRIX     #sigma = 1
circumplex_weight_matrix_05 = Dict_Object.ValueConstants.SCHWARTZ_VALUE_SIMILARITY_MATRIX_05  #sigma = 0.5
circumplex_weight_matrix_02 = Dict_Object.ValueConstants.SCHWARTZ_VALUE_SIMILARITY_MATRIX_02  #sigma = 0.2

# Normal distribution sigma for circumplex weights (same as paper)
SIGMA = 1.0

OUT_SIM = "/Proyecto/Value-disagreement/Python/Models/Inference/similarities/parent_child_profile_similarity.csv"
OUT_SUMMARY = "/Proyecto/Value-disagreement/Python/Models/Inference/similarities/parent_child_profile_similarity_summary.csv"

COMPUTE_EXTRAS = True  # prev + std

In [ ]:
# Helpers
def parse_list(x):
    """Parse list stored as string into Python list."""
    if pd.isna(x):
        return None
    if isinstance(x, list):
        return x
    if isinstance(x, (tuple, np.ndarray)):
        return list(x)
    s = str(x).strip()
    if not s:
        return None
    try:
        return ast.literal_eval(s)
    except Exception:
        try:
            return json.loads(s)
        except Exception:
            return None

def to_np(x):
    if x is None:
        return None
    arr = np.asarray(x, dtype=float)
    if arr.ndim != 1:
        arr = arr.ravel()
    return arr

def safe_norm(v):
    n = np.linalg.norm(v)
    return n if n > 0 else np.nan

def cosine_sim(a, b):
    na = safe_norm(a); nb = safe_norm(b)
    if not np.isfinite(na) or not np.isfinite(nb):
        return np.nan
    return float(np.dot(a, b) / (na * nb))

def circumplex_weight_matrix(sigma=1.0):
    if sigma == 0.5:
        W = circumplex_weight_matrix_05
    elif sigma == 0.2:
        W = circumplex_weight_matrix_02
    elif sigma == 1:
        W = circumplex_weight_matrix_1
    elif sigma == 2:
        W = circumplex_weight_matrix_2
    elif sigma == 5:
        W = circumplex_weight_matrix_5
    return W

W = circumplex_weight_matrix(SIGMA)

def weighted_cosine(a, b, W):
    num = float(a.T @ W @ b)
    den_a = float(a.T @ W @ a)
    den_b = float(b.T @ W @ b)
    if den_a <= 0 or den_b <= 0:
        return np.nan
    return num / math.sqrt(den_a * den_b)

def normalize_to_prob(v, eps=1e-12):
    if v is None:
        return None
    v = np.asarray(v, dtype=float)
    v = np.where(np.isfinite(v), v, 0.0)
    s = float(v.sum())
    if s <= 0:
        return None
    p = v / s
    p = np.clip(p, eps, 1.0)
    p = p / float(p.sum())
    return p

def jsd(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    p = np.clip(p, eps, 1.0); q = np.clip(q, eps, 1.0)
    p = p / p.sum(); q = q / q.sum()
    m = 0.5 * (p + q)

    def kl(a, b):
        return float(np.sum(a * (np.log2(a) - np.log2(b))))
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

def kendall_tau_tie_aware(a, b):
    ra = rankdata(a, method="average")
    rb = rankdata(b, method="average")
    tau, _ = kendalltau(ra, rb)
    return float(tau) if tau is not None else np.nan

In [ ]:
df = pd.read_csv(DATA_PATH, sep=SEP, dtype=str)

# Coerce numeric columns
for col in ["count_parent", "count_child", "agreement_fraction", "individual_kappa",
            "p_total_comments", "c_total_comments", "p_total_value_mentions", "c_total_value_mentions"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Parse vectors
for col in ["p_paper_vector","c_paper_vector","p_prob_vector","c_prob_vector"]:
    if col in df.columns:
        df[col] = df[col].apply(parse_list)

if COMPUTE_EXTRAS:
    for col in ["p_binary_vector","c_binary_vector","p_std_vector","c_std_vector"]:
        if col in df.columns:
            df[col] = df[col].apply(parse_list)

# numeric label
if "label" in df.columns:
    df["label_num"] = df["label"].astype("Int64")
    mapping = {"0": "disagree", "1":"neutral", "2":"agree"}
    df["label"] = df["label"].str.lower().map(mapping)

df.head(3)

In [ ]:
def compute_metrics_row(row):
    out = {}

    p_paper = to_np(row.get("p_paper_vector"))
    c_paper = to_np(row.get("c_paper_vector"))
    p_prob  = to_np(row.get("p_prob_vector"))
    c_prob  = to_np(row.get("c_prob_vector"))

    if p_paper is not None and c_paper is not None and len(p_paper)==NUM_VALUES and len(c_paper)==NUM_VALUES:
        a = p_paper#[VALUES_ORDER]
        b = c_paper#[VALUES_ORDER]
        out["paper_cos"] = cosine_sim(a, b)
        out["paper_wc"]  = weighted_cosine(a, b, W)
        out["paper_tau"] = kendall_tau_tie_aware(a, b)
    else:
        out["paper_cos"] = np.nan
        out["paper_wc"]  = np.nan
        out["paper_tau"] = np.nan

    if p_prob is not None and c_prob is not None and len(p_prob)==NUM_VALUES and len(c_prob)==NUM_VALUES:
        a = p_prob#[VALUES_ORDER]
        b = c_prob#[VALUES_ORDER]
        out["prob_cos"] = cosine_sim(a, b)
        out["prob_wc"]  = weighted_cosine(a, b, W)

        pN = normalize_to_prob(p_prob)
        cN = normalize_to_prob(c_prob)
        out["prob_jsd"] = jsd(pN, cN) if (pN is not None and cN is not None) else np.nan
    else:
        out["prob_cos"] = np.nan
        out["prob_wc"]  = np.nan
        out["prob_jsd"] = np.nan

    # extras: prev + std 
    if COMPUTE_EXTRAS:
        p_prev = to_np(row.get("p_binary_vector"))
        c_prev = to_np(row.get("c_binary_vector"))
        p_std  = to_np(row.get("p_std_vector"))
        c_std  = to_np(row.get("c_std_vector"))

        # prev (frecuencia) — mejor como shape similarity
        if p_prev is not None and c_prev is not None and len(p_prev)==NUM_VALUES and len(c_prev)==NUM_VALUES:
            a = p_prev#[CIRC_ORDER]
            b = c_prev#3[CIRC_ORDER]
            out["prev_cos"] = cosine_sim(a, b)
        else:
            out["prev_cos"] = np.nan

        # std (estabilidad) — diferencia L2 (qué tan distinto es el patrón de variabilidad)
        if p_std is not None and c_std is not None and len(p_std)==NUM_VALUES and len(c_std)==NUM_VALUES:
            a = p_std#[CIRC_ORDER]
            b = c_std#[CIRC_ORDER]
            out["std_l2"] = float(np.linalg.norm(a - b))
        else:
            out["std_l2"] = np.nan

    return pd.Series(out)

metrics = df.apply(compute_metrics_row, axis=1)
result = pd.concat([df, metrics], axis=1)

result[["paper_cos","paper_wc","paper_tau","prob_cos","prob_wc","prob_jsd"]].describe()

In [ ]:
len(result)

In [ ]:
result.to_csv(OUT_SIM, index=False)

In [ ]:
# Filters

# (A) Drop neutrals (keep agree vs disagree)
result = result[result["label_num"] != 1]

# (B) Require value signal (base filter)
#result = result[(result["p_total_value_mentions"] > 0) & (result["c_total_value_mentions"] > 0)]

# (C) Paper-style profile thresholds l ∈ {1,10,50,200,500}
l = 500
result = result[(result["p_total_value_mentions"] >= l) & (result["c_total_value_mentions"] >= l)]

# (D) Your discourse-activity filter (if you want it in the same place)
# result = result[(result["count_parent"] >= 200) & (result["count_child"] >= 200)]

result[["paper_cos","paper_wc","paper_tau","prob_cos","prob_wc","prob_jsd"]].describe()

In [ ]:
len(result) # A        (15722)

In [ ]:
len(result) # A + B    (15722)

In [ ]:
len(result) # A + C10  (15722)

In [ ]:
len(result) # A + C50  (15708)

In [ ]:
len(result) # A + C100  (15412)

In [ ]:
len(result) # A + C200  (14336)

In [ ]:
len(result) # A + C500  (10553)

In [ ]:
# Save outputs
result.to_csv(OUT_SIM, index=False)

summary_cols = ["paper_cos","paper_wc","paper_tau","prob_cos","prob_wc","prob_jsd"]
if COMPUTE_EXTRAS:
    summary_cols += ["prev_cos","std_l2"]
    
group_key = "label" if "label" in result.columns else ("label_num" if "label_num" in result.columns else None)

if group_key is not None:
    summary = (
        result.groupby(group_key)[summary_cols]
        .agg(["count","mean","std","median"])
        .reset_index()
    )
else:
    summary = pd.DataFrame()

summary.to_csv(OUT_SUMMARY, index=False)

OUT_SIM, OUT_SUMMARY, summary.head(10)

In [ ]:
# Hipòthesis testing
L_VALUES = [50, 100, 200, 500]
METRICS = {
    "paper_wc": "paper_wc",
    "prob_wc": "prob_wc"
}

BF_POS = 3.0
BF_NEG = 1/3

In [ ]:
# compute BF10
def compute_bf10(x_agree, x_disagree):
    # pingouin expects arrays
    res = pg.bayesfactor_ttest(
        x_agree.values,
        x_disagree.values,
        paired=False
    )
    return float(res["BF10"])

def compute_bf10_from_samples(x_agree, x_disagree, equal_var=False):
    # drop NaNs
    a = pd.to_numeric(x_agree, errors="coerce").dropna().values
    d = pd.to_numeric(x_disagree, errors="coerce").dropna().values

    if len(a) < 2 or len(d) < 2:
        return np.nan

    # t-test (Welch by default)
    t_stat, _ = ttest_ind(a, d, equal_var=equal_var)

    # pingouin expects t, nx, ny
    bf10 = pg.bayesfactor_ttest(t_stat, nx=len(a), ny=len(d), paired=False)
    return float(bf10)

In [ ]:
# Run BF analysis
rows = []

for l in L_VALUES:
    df = pd.read_csv(f"/Proyecto/Value-disagreement/Python/Models/Inference/similarities/parent_child_profile_similarity_A_C{l}.csv")

    for subreddit, df_sub in df.groupby("subreddit"):

        agree = df_sub[df_sub["label_num"] == 2]
        disagree = df_sub[df_sub["label_num"] == 0]

        # minimum N safeguard (like paper warnings)
        if len(agree) < 20 or len(disagree) < 20:
            continue

        for metric_name, col in METRICS.items():
            bf10 = compute_bf10_from_samples(agree[col], disagree[col], equal_var=False)

            mean_agree = agree[col].mean()
            mean_disagree = disagree[col].mean()

            rows.append({
                "subreddit": subreddit,
                "l": l,
                "metric": metric_name,
                "BF10": bf10,
                "n_agree": len(agree),
                "n_disagree": len(disagree),
                "mean_agree": mean_agree,
                "mean_disagree": mean_disagree,
                "direction": "disagree < agree" if mean_disagree < mean_agree else "disagree ≥ agree"
            })

bf_df = pd.DataFrame(rows)

bf_positive = bf_df[bf_df["BF10"] > BF_POS].sort_values("BF10", ascending=False)
bf_negative = bf_df[bf_df["BF10"] < BF_NEG].sort_values("BF10")
bf_all = bf_df.sort_values(["l", "subreddit", "metric"])

bf_positive, bf_negative, bf_all.head()

In [ ]:
bf_all.sort_values(['subreddit', 'l'])
#bf_negative[bf_negative["subreddit"]=="BlackLivesMatter"]

In [ ]:
bf_negative[bf_negative["direction"]=="disagree < agree"]

In [ ]:
# RESUL TABLEs

In [ ]:
# table 57
files = {
    50:  "/Proyecto/Value-disagreement/Python/Models/Inference/similarities/parent_child_profile_similarity_summary_A_C50.csv",
    100: "/Proyecto/Value-disagreement/Python/Models/Inference/similarities/parent_child_profile_similarity_summary_A_C100.csv",
    200: "/Proyecto/Value-disagreement/Python/Models/Inference/similarities/parent_child_profile_similarity_summary_A_C200.csv",
    500: "/Proyecto/Value-disagreement/Python/Models/Inference/similarities/parent_child_profile_similarity_summary_A_C500.csv",
}

rows = []
for l, fn in files.items():
    df = pd.read_csv(fn)

    df = df[df["label"].isin(["agree", "disagree"])]

    # extraer medias (prob_wc.1) y conteos (prob_wc)
    agree = df[df["label"]=="agree"].iloc[0]
    disagree = df[df["label"]=="disagree"].iloc[0]

    mean_agree = float(agree["prob_wc.1"])
    mean_disagree = float(disagree["prob_wc.1"])

    rows.append({
        "ℓ": l,
        "N_agree": int(agree["prob_wc"]),
        "N_disagree": int(disagree["prob_wc"]),
        "Mean(sim_agree)": mean_agree,
        "Mean(sim_disagree)": mean_disagree,
        "Δ (agree − disagree)": mean_agree - mean_disagree
    })

table_57 = pd.DataFrame(rows).sort_values("ℓ")
table_57

In [ ]:
# tabla 58
bf_all

In [ ]:
def evidence_label(bf10: float) -> str:
    if bf10 > 3:
        return "H₁"
    if bf10 < (1/3):
        return "H₀"
    return "Inconcluso"

# Etiqueta final “Evidencia” (regla simple y transparente):
# - si hay al menos un H1 -> "H₁"
# - si hay al menos un H0 y no hay H1 -> "H₀"
# - si todo inconcluso -> "Inconcluso"
def overall_evidence(row):
    if row["n_H1"] > 0:
        return "H₁"
    if row["n_H0"] > 0:
        return "H₀"
    return "Inconcluso"

In [ ]:
df = bf_all.copy()

df["evidence"] = df["BF10"].apply(evidence_label)

summary_58_both = (
    df[df["metric"].isin(["paper_wc", "prob_wc"])]
      .groupby(["subreddit", "metric"])
      .agg(
          l_min=("l", "min"),
          l_max=("l", "max"),
          bf10_min=("BF10", "min"),
          bf10_max=("BF10", "max"),
          bf10_median=("BF10", "median"),
          n_tests=("BF10", "size"),
          n_H0=("evidence", lambda s: (s == "H₀").sum()),
          n_inconclusive=("evidence", lambda s: (s == "Inconcluso").sum()),
          n_H1=("evidence", lambda s: (s == "H₁").sum()),
      )
      .reset_index()
)

summary_58_both["Evidencia"] = summary_58_both.apply(overall_evidence, axis=1)
summary_58_both["ℓ"] = summary_58_both.apply(lambda r: f'{int(r["l_min"])}–{int(r["l_max"])}', axis=1)
summary_58_both["BF10 (rango)"] = summary_58_both.apply(lambda r: f'{r["bf10_min"]:.3f}–{r["bf10_max"]:.3f}', axis=1)

table_58_both = summary_58_both[[
    "metric", "subreddit", "ℓ", "BF10 (rango)", "bf10_median", "n_tests", "n_H0", "n_inconclusive", "n_H1", "Evidencia"
]].rename(columns={
    "metric": "Métrica",    
    "subreddit": "Subreddit",
    "bf10_median": "BF10 mediana",
    "n_tests": "N tests",
    "n_H0": "N(BF10<1/3)",
    "n_inconclusive": "N(1/3–3)",
    "n_H1": "N(BF10>3)",
})

table_58_both["BF10 mediana"] = table_58_both["BF10 mediana"].round(3)
table_58_both.sort_values('Métrica')

In [ ]:
df = bf_all.copy()

# Nos quedamos solo con la métrica principal
df = df[df["metric"] == "prob_wc"].copy()

# Añadimos etiqueta de evidencia por fila
df["evidence"] = df["BF10"].apply(evidence_label)

# Agregación por subreddit
summary_58 = (
    df.groupby("subreddit")
      .agg(
          l_min=("l", "min"),
          l_max=("l", "max"),
          bf10_min=("BF10", "min"),
          bf10_max=("BF10", "max"),
          bf10_median=("BF10", "median"),
          n_tests=("BF10", "size"),
          n_H0=("evidence", lambda s: (s == "H₀").sum()),
          n_inconclusive=("evidence", lambda s: (s == "Inconcluso").sum()),
          n_H1=("evidence", lambda s: (s == "H₁").sum()),
      )
      .reset_index()
)

summary_58["Evidencia"] = summary_58.apply(overall_evidence, axis=1)

# Formato bonito: rangos como texto
summary_58["ℓ"] = summary_58.apply(lambda r: f'{int(r["l_min"])}–{int(r["l_max"])}', axis=1)
summary_58["BF10 (prob_wc)"] = summary_58.apply(lambda r: f'{r["bf10_min"]:.3f}–{r["bf10_max"]:.3f}', axis=1)

# Selección final de columnas para el main text
table_58_main = summary_58[[
    "subreddit", "ℓ", "BF10 (prob_wc)", "bf10_median", "n_tests", "n_H0", "n_inconclusive", "n_H1", "Evidencia"
]].rename(columns={
    "subreddit": "Subreddit",
    "bf10_median": "BF10 mediana",
    "n_tests": "N tests",
    "n_H0": "N(BF10<1/3)",
    "n_inconclusive": "N(1/3–3)",
    "n_H1": "N(BF10>3)",
})

# Redondeo
table_58_main["BF10 mediana"] = table_58_main["BF10 mediana"].round(3)

table_58_main

In [ ]:
# Schwartz HIGHER DIMs Analysis
df = pd.read_csv(DATA_PATH, sep=SEP, dtype=str)
df

In [ ]:
for col in ["label", "p_total_value_mentions", "c_total_value_mentions"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
df.dtypes

In [ ]:
def parse_list_cell(x):
    if isinstance(x, list):
        return x
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    if s == "":
        return None

    # JSON (p.ej. ["universalism", ...])
    if s.startswith("[") and s.endswith("]"):
        try:
            return json.loads(s)
        except Exception:
            pass

    # Python literal
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            out = ast.literal_eval(s)
            if isinstance(out, (list, tuple)):
                return list(out)
        except Exception:
            pass

    # Fallback
    if "," in s:
        return [t.strip() for t in s.split(",") if t.strip()]
    return [t.strip() for t in s.split() if t.strip()]

def norm_value_name(v):
    s = str(v).strip().lower()
    s = s.replace("_", "-")
    s = re.sub(r"\s+", "", s)
    return s

VALS = {
    "universalism": "universalism",
    "benevolence": "benevolence",
    "conformity": "conformity",
    "tradition": "tradition",
    "security": "security",
    "power": "power",
    "achievement": "achievement",
    "hedonism": "hedonism",
    "stimulation": "stimulation",
    "self-direction": "self-direction",
    "selfdirection": "self-direction",
}

OPEN = ["self-direction", "stimulation", "hedonism"]
CONS = ["security", "conformity", "tradition"]
ST   = ["universalism", "benevolence"]
SE   = ["power", "achievement"]

def build_index_map(values_order):
    order_list = parse_list_cell(values_order)
    if not order_list:
        raise ValueError("values_order no se pudo parsear.")
    idx_map = {}
    for i, raw in enumerate(order_list):
        key = norm_value_name(raw)
        key = VALS.get(key, key)
        idx_map[key] = i
    return idx_map

def parse_vector_cell(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return np.array(x, dtype=float)
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    if s == "":
        return None

    # JSON primero (rápido y compatible con comillas dobles)
    if s.startswith("[") and s.endswith("]"):
        try:
            return np.array(json.loads(s), dtype=float)
        except Exception:
            pass

    # Python literal
    try:
        out = ast.literal_eval(s)
        return np.array(out, dtype=float)
    except Exception:
        # fallback split coma
        parts = [p.strip() for p in s.split(",") if p.strip()]
        try:
            return np.array([float(p) for p in parts], dtype=float)
        except Exception:
            return None

def axis_from_vector(vec, idx_map):
    def mean_group(group):
        idxs = [idx_map[norm_value_name(g)] for g in group]
        return float(np.mean(vec[idxs]))

    apertura = mean_group(OPEN)
    conserv  = mean_group(CONS)
    st       = mean_group(ST)
    se       = mean_group(SE)

    e1 = apertura - conserv
    e2 = st - se
    return e1, e2

In [ ]:
# Axes and distances
def add_axes_and_distances(df_in, parent_vec_col, child_vec_col, suffix):
    out = df_in.copy()

    vo = out["values_order"].dropna().iloc[0]
    idx_map = build_index_map(vo)

    p_vec = out[parent_vec_col].apply(parse_vector_cell)
    c_vec = out[child_vec_col].apply(parse_vector_cell)

    p_e1 = np.full(len(out), np.nan)
    p_e2 = np.full(len(out), np.nan)
    c_e1 = np.full(len(out), np.nan)
    c_e2 = np.full(len(out), np.nan)

    for i, (pv, cv) in enumerate(zip(p_vec, c_vec)):
        if pv is None or cv is None:
            continue
        pe1, pe2 = axis_from_vector(pv, idx_map)
        ce1, ce2 = axis_from_vector(cv, idx_map)
        p_e1[i], p_e2[i], c_e1[i], c_e2[i] = pe1, pe2, ce1, ce2

    out[f"p_e1_{suffix}"] = p_e1
    out[f"p_e2_{suffix}"] = p_e2
    out[f"c_e1_{suffix}"] = c_e1
    out[f"c_e2_{suffix}"] = c_e2

    out[f"dist_e1_{suffix}"] = np.abs(out[f"p_e1_{suffix}"] - out[f"c_e1_{suffix}"])
    out[f"dist_e2_{suffix}"] = np.abs(out[f"p_e2_{suffix}"] - out[f"c_e2_{suffix}"])
    out[f"dist_2d_{suffix}"] = np.sqrt(
        (out[f"p_e1_{suffix}"] - out[f"c_e1_{suffix}"])**2 +
        (out[f"p_e2_{suffix}"] - out[f"c_e2_{suffix}"])**2
    )
    return out
df_axes = df.copy()

if {"p_prob_vector","c_prob_vector"}.issubset(df_axes.columns):
    df_axes = add_axes_and_distances(df_axes, "p_prob_vector", "c_prob_vector", "prob")

if {"p_paper_vector","c_paper_vector"}.issubset(df_axes.columns):
    df_axes = add_axes_and_distances(df_axes, "p_paper_vector", "c_paper_vector", "paper")

In [ ]:
# Tabla 57 for HIGHER DIMs
def table_57_axes(df_in, suffix, ls=(50,100,200,500)):
    metric_cols = [f"dist_e1_{suffix}", f"dist_e2_{suffix}", f"dist_2d_{suffix}"]
    rows = []

    for l in ls:
        d = df_in[df_in["label"].isin([0,2])].copy()

        if {"p_total_value_mentions","c_total_value_mentions"}.issubset(d.columns):
            d = d[(d["p_total_value_mentions"] >= l) & (d["c_total_value_mentions"] >= l)]

        for col in metric_cols:
            m = d.groupby("label")[col].mean()
            n = d.groupby("label")[col].count()
            mean_dis = float(m.get(0, np.nan))
            mean_agr = float(m.get(2, np.nan))
            rows.append({
                "ℓ": l,
                "metric": col,
                "N_agree": int(n.get(2, 0)),
                "N_disagree": int(n.get(0, 0)),
                "mean_agree": mean_agr,
                "mean_disagree": mean_dis,
                "Δ (disagree - agree)": mean_dis - mean_agr
            })

    out = pd.DataFrame(rows)
    out["metric"] = out["metric"].replace({
        f"dist_e1_{suffix}": "dist_e1 (Open - Conserv)",
        f"dist_e2_{suffix}": "dist_e2 (ST - SE)",
        f"dist_2d_{suffix}": "dist_2d (E1,E2)"
    })
    return out

table_57_prob  = table_57_axes(df_axes, "prob")  if f"dist_e1_prob"  in df_axes.columns else None
table_57_paper = table_57_axes(df_axes, "paper") if f"dist_e1_paper" in df_axes.columns else None

In [ ]:
table_57_prob

In [ ]:
table_57_paper

In [ ]:
# BF10 por subreddit x ℓ para ejes (como tu Tabla 5.8)
def bf10_by_subreddit(df_in, suffix, ls=(50,100,200,500)):
    import pingouin as pg

    metric_cols = [f"dist_e1_{suffix}", f"dist_e2_{suffix}", f"dist_2d_{suffix}"]
    rows = []

    for l in ls:
        d = df_in[df_in["label"].isin([0,2])].copy()
        if "p_total_value_mentions" in d.columns and "c_total_value_mentions" in d.columns:
            d = d[(d["p_total_value_mentions"] >= l) & (d["c_total_value_mentions"] >= l)]

        for subreddit, g in d.groupby("subreddit"):
            for col in metric_cols:
                x = g[g["label"]==0][col].dropna()  # disagree
                y = g[g["label"]==2][col].dropna()  # agree
                if len(x) < 5 or len(y) < 5:
                    continue

                res = pg.ttest(x, y, correction=True)  # Welch
                bf10 = float(res["BF10"].iloc[0])

                rows.append({
                    "subreddit": subreddit,
                    "l": l,
                    "metric": col,
                    "BF10": bf10,
                    "n_disagree": len(x),
                    "n_agree": len(y),
                    "mean_disagree": float(x.mean()),
                    "mean_agree": float(y.mean()),
                    "direction": "disagree > agree" if x.mean() > y.mean() else "disagree <= agree"
                })

    out = pd.DataFrame(rows)
    out["metric"] = out["metric"].replace({
        f"dist_e1_{suffix}": "dist_e1 (Open - Conserv)",
        f"dist_e2_{suffix}": "dist_e2 (ST - SE)",
        f"dist_2d_{suffix}": "dist_2d (E1,E2)"
    })
    return out.sort_values(["subreddit","l","metric"])

# Ejemplo:
bf_prob = bf10_by_subreddit(df_axes, "prob")
bf_paper = bf10_by_subreddit(df_axes, "paper")

In [ ]:
bf_prob

In [ ]:
bf_paper

In [ ]:
# BF10 by SubReddit

# filter
metric_name = "dist_2d (E1,E2)"
d = bf_prob[bf_prob["metric"] == metric_name].copy()

mat = d.pivot(index="subreddit", columns="l", values="BF10")

# table 58 order
order_subs = ["BlackLivesMatter", "Brexit", "Republican", "climate", "democrats"]
mat = mat.loc[[s for s in order_subs if s in mat.index]]

# log10 transformation for visualizacion
eps = 1e-12 # in case 0
logmat = np.log10(mat.astype(float) + eps)

# Plot heatmap
fig, ax = plt.subplots(figsize=(1.2 * len(mat.columns) + 3.5, 0.6 * len(mat.index) + 2.5))
im = ax.imshow(logmat.values, aspect="auto")

# ticks
ax.set_xticks(np.arange(len(mat.columns)))
ax.set_xticklabels(mat.columns.tolist())
ax.set_yticks(np.arange(len(mat.index)))
ax.set_yticklabels(mat.index.tolist())

ax.set_xlabel("ℓ (mínimo de menciones de valores)")
ax.set_ylabel("Subreddit")
ax.set_title("Heatmap de evidencia Bayesiana por dominio\nlog10(BF10) para dist_2d (E1,E2)")

# Colorbar
cbar = fig.colorbar(im, ax=ax)
cbar.ax.set_ylabel("log10(BF10)", rotation=90)

for i in range(logmat.shape[0]):
    for j in range(logmat.shape[1]):
        bf = mat.iloc[i, j]
        txt = "NA" if pd.isna(bf) else f"{bf:.2f}"
        ax.text(j, i, txt, ha="center", va="center")

plt.tight_layout()

#plt.savefig("bf10_heatmap_dist2d.png", dpi=300, bbox_inches="tight")
#plt.savefig("bf10_heatmap_dist2d.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Cualitative Examples
df = result.copy()
df

In [ ]:
for col in ["label_num", "p_total_value_mentions", "c_total_value_mentions"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
df.dtypes

In [ ]:
# Filtro A (sin neutrales)
df = df[df["label_num"].isin([0,2])&(df["p_total_value_mentions"] >= 100) & (df["c_total_value_mentions"] >= 100)]
len(df)

In [ ]:
df["label"]

In [ ]:
metric = "prob_wc"
def parse_vec(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return np.array(x, dtype=float)

    # string tipe "[0.1, 0.2, ...]"
    if isinstance(x, str):
        x = x.strip()
        if x == "":
            return None
        try:
            return np.array(ast.literal_eval(x), dtype=float)
        except Exception:
            return None

    # NaN real
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None

    return None

df["p_vec"] = df["p_prob_vector"].apply(parse_vec)
df["c_vec"] = df["c_prob_vector"].apply(parse_vec)

In [ ]:
# Crear la distancia (euclídea) entre perfiles probabilísticos
df["dist_2d_prob"] = [
    np.linalg.norm(p - c) if (p is not None and c is not None) else np.nan
    for p, c in zip(df["p_vec"], df["c_vec"])
]

# Alta distancia (percentil 95)
thr_high = df[metric].quantile(0.95)
high_dist_agree = df[(df[metric] >= thr_high) & (df["label_num"] == 2)]

# Baja distancia (percentil 5)
thr_low  = df[metric].quantile(0.05)
low_dist_disagree = df[(df["dist_2d_prob"] <= thr_low) & (df["label_num"] == 0)]

In [ ]:
high_dist_agree

In [ ]:
low_dist_disagree

In [ ]:
high_examples

In [ ]:
low_examples